<a href="https://colab.research.google.com/github/AshACxx/AshsProjects/blob/main/MongoStuff.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
from pymongo import MongoClient
from google.colab import userdata

client = MongoClient(userdata.get("MONGO_URI").strip())
client.admin.command("ping")
print("Connected!")

Connected!


In [16]:
!pip install -q pyspark==3.5.1 pymongo \
dnspython certifi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 8.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.


In [17]:
import pyspark
print(pyspark.__version__)
# must print 3.5.1

3.5.1


In [19]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark = (
SparkSession.builder
.appName("MongoSparkDemo")
.config("spark.jars.packages",
"org.mongodb.spark:mongo-spark-"
"connector_2.12:10.4.0")
.config("spark.mongodb.read.connection.uri",
MONGO_URI)
.config("spark.mongodb.write.connection.uri",
MONGO_URI)
.getOrCreate()
)
print(spark.version) # 3.5.1

3.5.1


In [20]:
data = [("Alice","Electronics",1200,"Dublin"),
("Bob","Books",45,"Cork"),
("Cara","Electronics",800,"Dublin"),
("Dan","Clothing",150,"Galway")]
df = spark.createDataFrame(data,["customer","category","amount","city"])
(df.write.format("mongodb")
.mode("overwrite")
.option("database", "shop")
.option("collection", "sales")
.save())

In [23]:
sales = (spark.read.format("mongodb")
.option("database", "shop")
.option("collection", "sales").load())
sales.printSchema()
sales.show()

root
 |-- _id: string (nullable = true)
 |-- amount: long (nullable = true)
 |-- category: string (nullable = true)
 |-- city: string (nullable = true)
 |-- customer: string (nullable = true)

+--------------------+------+-----------+------+--------+
|                 _id|amount|   category|  city|customer|
+--------------------+------+-----------+------+--------+
|6abd6bfa69c6c53c7...|  1200|Electronics|Dublin|   Alice|
|6abd6bfa69c6c53c7...|    45|      Books|  Cork|     Bob|
|6abd6bfa69c6c53c7...|   800|Electronics|Dublin|    Cara|
|6abd6bfa69c6c53c7...|   150|   Clothing|Galway|     Dan|
+--------------------+------+-----------+------+--------+

